In [0]:
%run ../config/feat_squad2_config_sqlserver

In [0]:
%run ../config/feat_squad2_config_adls

In [0]:
%run ../utils/feat_squad2_utils

In [0]:
# Importar bibliotecas
import time

In [0]:
# Configurar as variáveis
folder_name = "vendas_raw/"
entity_name = "enderecos"
file_name_contains = f"ecommerce_{entity_name}.parquet"
table_name = f"squad2.ecommerce_{entity_name}"
check_interval = 10 
ingestion_log = []
processed_files = set()
snapshot_id = 0

In [0]:
# Configuração do logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

log = logging.getLogger("pipeline_ingestion")

logging.getLogger("azure").setLevel(logging.WARNING)
logging.getLogger("azure.identity").setLevel(logging.WARNING)
logging.getLogger("azure.core").setLevel(logging.WARNING)

In [0]:
while True:
    snapshot_id += 1

    log.info(f"Iniciando ciclo {snapshot_id} para entidade {entity_name}. Verificando novos arquivos")

    all_files = list_files(
                        container_client=container_client,
                        folder_name=folder_name,
                        file_name_contains=file_name_contains
    )

    list_files_to_ingest = [
        file_path
        for file_path in all_files
        if file_path not in processed_files
    ]
    
    num_files = len(list_files_to_ingest)

    if num_files == 0:
        log.info(
            f"Não há novos arquivos para ingerir. Encerrando ciclo {snapshot_id}."
        )

        time.sleep(check_interval)
        continue
    
    log.info(f"{num_files} novos arquivos encontrados. Iniciando ingestão.")

    for file_path in list_files_to_ingest:
        try:
            log.info(f"Processando arquivo {file_path}")

            df_snapshot = read_parquet_to_spark_df(
                                                file_path = file_path
            )

            df_snapshot_count = df_snapshot.count()

            write_sql_server(
                            df = df_snapshot,
                            table_name = table_name,
                            jdbc_hostname = jdbc_hostname,
                            jdbc_database = jdbc_database,
                            jdbc_username = jdbc_username,
                            jdbc_password = jdbc_password,
                            mode="append"
                        )
            
            processed_files.add(file_path)

            log_ingestion(
                        snapshot_id = snapshot_id,
                        arquivo = file_path,
                        registros = df_snapshot_count,
                        status = "SUCESSO",
                        mensagem = "Arquivo ingerido com sucesso"
            )

            log.info(f"Arquivo {file_path} ingerido com sucesso.")

        except Exception as e:

            log_ingestion(
                        snapshot_id = snapshot_id,
                        arquivo = file_path,
                        registros = 0,
                        status = "FALHA",
                        mensagem = str(e)
            )

            log.error(f"Erro no arquivo {file_path}. Mensagem: {str(e)}")

    log.info(f"Ciclo {snapshot_id} finalizado.")
    time.sleep(check_interval)